In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches


from PIL import Image

# Will be used for the dark spots in the dataset
from scipy import ndimage

# Fits a straight line compared to the longitude vs time
from scipy.stats import lineregress


In [ ]:
IMAGE = "../Data"

# The Dataset size
Days = 9

This gets the center of the Solar Disk and the radius

In [ ]:
'''
Function: Gets the Center of the Sun and calculated the radius of the Sun in the image

Input:
    Image

Return: 
    X coordinate
    Y coordinate
    Radius

Sources:
    CMPUT 206 Notes

'''

def getCenter(img_arr):

    # Since HMI orange images are the strongest in the RED channel
    # Of the image, it will be best used to differentiate
    # The black space and orange Sun

    red_img = img_arr[:,:,0].astype(float)

    threshold = 30

    # Gets a mask to make sure only the sun is considered for center
    # Creates a binary grid:
    # 1: If it is the sun
    # 0: If it is outer space
    mask = red > threshold

    rows = []
    # Check every row which contains a pixel of the sun
    # If a row has a sun in it then we keep the position of the row in an array
    for r in range(mask.shape[0]):
        if 1 in mask[r, :]:
            rows.append(r)
    rows = np.array(rows)

    # Goes through each column checking every pixel
    # Checks if it contains any part of the suns pixel
    # If it does store the positioning
    cols = []
    for c in range(mask.shape[1]):
        if 1 in mask[:,c]:
            cols.append(c)
    cols = np.array(cols)

    row_min = rows[0]
    row_max = rows[-1]

    col_min = cols[0]
    col_max = cols[-1]

    # (Max + Min) / 2 
    # Gets the center coordinates

    y = (row_max + row_min)/ 2.0
    x = (col_max + col_min)/ 2.0

    # Radius
    vertD = row_max - row_min
    horzD = col_max - col_min

    R = (vertD + horzD) / 4.0

    return x,y,R

Gets all the Sunspots and returns the coordinates of the sunspots

In [ ]:
def getSunspots(img_arr, x,y,R):

    # Gets the different color channels RGB for the image
    rImg = img_array[:,:,0].astype(float)
    gImg = img_array[:,:,1].astype(float)
    bImg = img_array[:,:,2].astype(float)

    # Since sunspots are dark in every color channel
    # the average brightness can be used as a threshold
    avgB = (rImg + gImg + bImg) / 3.0